In [1]:
import sys
sys.path.append("..")

from pipeline.constants import *
from pipeline.utils import register_tables_in_spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/06 14:41:11 WARN Utils: Your hostname, bbas, resolves to a loopback address: 127.0.1.1; using 10.236.167.43 instead (on interface wlp0s20f3)
26/10/06 14:41:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 14:41:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
register_tables_in_spark(SPARK_SESSION, PARQUET_DIR)

26/10/06 14:42:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
SPARK_SESSION.sql("show tables;").show(n=50, truncate=False)

+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|         |call_center           |true       |
|         |catalog_page          |true       |
|         |catalog_returns       |true       |
|         |catalog_sales         |true       |
|         |customer              |true       |
|         |customer_address      |true       |
|         |customer_demographics |true       |
|         |date_dim              |true       |
|         |household_demographics|true       |
|         |income_band           |true       |
|         |inventory             |true       |
|         |item                  |true       |
|         |promotion             |true       |
|         |reason                |true       |
|         |ship_mode             |true       |
|         |store                 |true       |
|         |store_returns         |true       |
|         |store_sales           |true       |
|         |ti

In [5]:
SPARK_SESSION.sql("select distinct s_state from store;").show(n=50, truncate=False)

+-------+
|s_state|
+-------+
|TN     |
+-------+



In [ ]:
query = """with customer_total_return as
(select sr_customer_sk as ctr_customer_sk
,sr_store_sk as ctr_store_sk
,sum(SR_FEE) as ctr_total_return
from store_returns
,date_dim
where sr_returned_date_sk = d_date_sk
and d_year =2000
group by sr_customer_sk
,sr_store_sk)
 select  c_customer_id
from customer_total_return ctr1
,store
,customer
where ctr1.ctr_total_return > (select avg(ctr_total_return)*1.2
from customer_total_return ctr2
where ctr1.ctr_store_sk = ctr2.ctr_store_sk)
and s_store_sk = ctr1.ctr_store_sk
and s_state = 'TN'
and ctr1.ctr_customer_sk = c_customer_sk
order by c_customer_id
limit 100;
"""

res = SPARK_SESSION.sql(query)

In [ ]:
n = res.count()
print(f"Number of rows returned: {n}")
print(res.show(n=n, truncate=False))